In [1]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import re
import chromadb
import pandas as pd
from scripts.pplx_embed import PplxEmbedFunction
from dotenv import load_dotenv
from mlx_lm import generate, load

load_dotenv()

/Users/sergey/Desktop/Moodle_RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
PPLX_CHROMA_DIR = "/Users/sergey/Desktop/Moodle_RAG/scripts/Notebooks/chroma_db_pplx"
PPLX_COLLECTION = "moodle_docs_pplx"

pplx_embed = PplxEmbedFunction()
client = chromadb.PersistentClient(path=PPLX_CHROMA_DIR)
collection = client.get_collection(
    name=PPLX_COLLECTION,
    embedding_function=pplx_embed,
)
print(f"📦 Чанков в коллекции: {collection.count()}")

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1927.94it/s, Materializing param=norm.weight]                             


📦 Чанков в коллекции: 3395


In [3]:
# LLM для ответа
model, tokenizer = load("mlx-community/Qwen2.5-7B-Instruct-4bit")

Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 140329.87it/s]


In [4]:
def build_context(user_query: str, k: int = 5):
    """Поиск по PPLX коллекции. Без перевода — модель мультиязычная."""
    results = collection.query(query_texts=[user_query], n_results=k)

    context_blocks = []
    for i, (doc, meta, dist) in enumerate(zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ), 1):
        context_blocks.append(
            f"[{i}]\n"
            f"source: {meta.get('source', 'unknown')}\n"
            f"distance: {dist:.4f}\n"
            f"text:\n{doc}"
        )

    context = "\n\n---\n\n".join(context_blocks)
    return context, results

In [5]:
def generate_answer(user_query: str, context: str, recent_history=None):
    system_prompt = """
    <ROLE_DEFINITION>
    You are the official Moodle documentation assistant acting as a technical consultant.
    </ROLE_DEFINITION>

    <MAIN_TASK_GUIDELINES>
    Your task is to provide precise, formal, and verifiable answers strictly based on the provided CONTEXT.
    You must directly answer the user's question without adding external knowledge.
    If the answer is not present in the CONTEXT, respond exactly: "Not found in the documentation".
    Do not make assumptions, interpretations, or extrapolations beyond the CONTEXT.
    The response must be structured and concise.
    </MAIN_TASK_GUIDELINES>

    <IMPORTANT_LANGUAGE_GUIDELINES>
    Determine the language of the user's query and use THAT SAME language for:
    - all actions,
    - all search formulations,
    - the final answer,
    - all textual fields and outputs.

    If the query is in Russian — all fields and responses must be strictly in Russian.
    If the query is in English — all fields and responses must be strictly in English.
    </IMPORTANT_LANGUAGE_GUIDELINES>

    <OUTPUT_FORMAT_REQUIREMENTS>
    The ending section is mandatory and must always be included:

    - source: [list of source filenames from CONTEXT]
    </OUTPUT_FORMAT_REQUIREMENTS>
    """.strip()

    messages = [{"role": "system", "content": system_prompt}]

    if recent_history:
        for m in recent_history:
            messages.append({"role": m["role"], "content": m["content"]})

    messages.append(
        {
            "role": "user",
            "content": (
                f"QUESTION:\n{user_query}\n\n"
                f"CONTEXT:\n{context}"
            ),
        }
    )

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    answer = generate(model, tokenizer, prompt=prompt, max_tokens=550)
    return answer

In [6]:
def retrieval_query_with_context(query: str, chat_history: list):
    q = query.strip()
    if len(q.split()) <= 4:
        prev_users = [m["content"] for m in chat_history if m["role"] == "user"]
        if prev_users:
            return prev_users[-1] + "\n" + q
    return q

In [7]:
def continual_chat(k: int = 5, history_turns: int = 3):
    print("Начните общение с ИИ! Введите 'выход', чтобы завершить разговор.")
    chat_history = []

    while True:
        query = input("ВЫ: ").strip()
        if query.lower() in ("выход", "exit", "quit"):
            break

        recent_history = chat_history[-2 * history_turns :]

        rq = retrieval_query_with_context(query, chat_history)
        context, results = build_context(rq, k=k)

        answer = generate_answer(query, context, recent_history=recent_history)

        print("\n✅ Ответ AI:")
        print(answer)

        chat_history.append({"role": "user", "content": query})
        chat_history.append({"role": "assistant", "content": answer})


if __name__ == "__main__":
    continual_chat()

Начните общение с ИИ! Введите 'выход', чтобы завершить разговор.


/Users/sergey/.cache/huggingface/modules/transformers_modules/perplexity_hyphen_ai/pplx_hyphen_embed_hyphen_v1_hyphen_0_dot_6b/1dc2ea99a948a2f17b103949ad02b0194a20c0a8/modeling.py:62: FutureWarning: `input_embeds` is deprecated and will be removed in version 5.6.0 for `create_causal_mask`. Use `inputs_embeds` instead.
  "full_attention": create_causal_mask(



✅ Ответ AI:
Чтобы создать новый курс в Moodle, выполните следующие шаги:

1. Откройте раздел Site administration и перейдите к Courses > Manage courses and categories.
2. Нажмите на New course в категории, где вы хотите создать курс.
3. Выберите категорию, в которой вы хотите разместить курс.
4. Нажмите на ссылку New course.
5. Установите параметры курса и выберите, хотите ли вы сохранить и вернуться к настройкам курса или сохранить и перейти к следующему экрану.
6. Если вы выбрали "Save and display", назначьте студентов/преподавателей для этого курса.

Источник: 403__en__Adding_a_new_course.md

source: [403__en__Adding_a_new_course.md]

✅ Ответ AI:
Чтобы настроить систему оценок в Moodle, выполните следующие шаги:

1. Войдите в раздел Site administration и перейдите к Advanced features. Включите функцию Outcomes.
2. Перейдите в ваш курс и выберите Administration > Course administration > Grades > Scales. Создайте необходимую шкалу оценок.
3. Перейдите в Administration > Course admini

# Максимально простой замер базовых метрик RAGа

In [8]:
eval_items = [
    {
        "question": "Как создать новый курс в Moodle?",
        "relevant_sources": ["Adding_a_new_course"],
    },
    {
        "question": "Как настроить систему оценок в Moodle?",
        "relevant_sources": ["Grades", "Gradebook"],
    },
    {
        "question": "Как просмотреть журналы активности пользователей?",
        "relevant_sources": ["Activity_report"],
    },
]


def retrieval_hit_mrr(results, relevant_sources, k=5):
    """Hit@k и MRR по source из метаданных PPLX коллекции."""
    metas = results["metadatas"][0][:k]
    hit = 0
    mrr = 0.0

    for rank, meta in enumerate(metas, start=1):
        source = meta.get("source", "")
        if any(rs.lower() in source.lower() for rs in relevant_sources):
            hit = 1
            mrr = 1.0 / rank
            break

    return hit, mrr


def llm_judge_faithfulness(question, context, answer):
    judge_prompt = f"""
You are a strict faithfulness judge.
Task: check whether ANSWER is fully supported by CONTEXT only.

Return EXACTLY in this format:
FAITHFUL: yes/no
CITATIONS:
- "short quote 1 from CONTEXT"
- "short quote 2 from CONTEXT"

Rules:
- If any claim in ANSWER is not in CONTEXT => FAITHFUL: no
- Quotes must be verbatim from CONTEXT
- Max 2 quotes

QUESTION:
{question}

CONTEXT:
{context}

ANSWER:
{answer}
""".strip()

    messages = [{"role": "user", "content": judge_prompt}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    judge_raw = generate(model, tokenizer, prompt=prompt, max_tokens=180)

    faithful = 1 if "FAITHFUL: yes" in judge_raw else 0
    citations = re.findall(r'-\s*"([^"]+)"', judge_raw)
    return faithful, citations, judge_raw


rows = []

for item in eval_items:
    q = item["question"]
    relevant = item["relevant_sources"]

    context, results = build_context(q, k=5)
    answer = generate_answer(q, context)

    hit5, mrr5 = retrieval_hit_mrr(results, relevant, k=5)
    faithful, judge_citations, judge_raw = llm_judge_faithfulness(q, context, answer)

    rows.append(
        {
            "question": q,
            "hit@5": hit5,
            "mrr@5": round(mrr5, 4),
            "faithfulness_llm_judge": faithful,
            "faithfulness_citations": judge_citations,
            "relevant_sources": relevant,
            "answer": answer[:200] + "...",
        }
    )

df_eval = pd.DataFrame(rows)
df_eval

,question,hit@5,mrr@5,faithfulness_llm_judge,faithfulness_citations,relevant_sources,answer
0,Как создать новый курс в Moodle?,1,1.0,1,"[From the Site administration link, click Cour...",[Adding_a_new_course],"Чтобы создать новый курс в Moodle, выполните с..."
1,Как настроить систему оценок в Moodle?,1,1.0,1,[Administration > Site administration > Advanc...,"[Grades, Gradebook]","Чтобы настроить систему оценок в Moodle, выпол..."
2,Как просмотреть журналы активности пользователей?,1,1.0,1,"[can be viewed by managers, teachers and non-e...",[Activity_report],Чтобы просмотреть журналы активности пользоват...
